# Behavioral Baseline (A100)

Measure baseline behavior before activation probing.

[实验设置与运行说明](../../../docs/experiment_notes/00_dataset_and_behavior/f825a39ae9f1768c.md)


In [1]:
# Run this helper cell once. Every later code cell uses %%cell_progress.
import re
import threading
import time
from IPython.core.magic import register_cell_magic
from tqdm.auto import tqdm

_SETUP_PROGRESS = tqdm(total=1, desc='Initialize progress display', unit='cell')
_ACTIVE_CELL_PROGRESS = None

def cell_stage(message):
    """Update the stage text shown by the active cell progress bar."""
    if _ACTIVE_CELL_PROGRESS is not None:
        _ACTIVE_CELL_PROGRESS['stage'] = str(message)
        elapsed = time.monotonic() - _ACTIVE_CELL_PROGRESS['started']
        _ACTIVE_CELL_PROGRESS['bar'].set_postfix_str(
            f"{message} | elapsed {elapsed:.0f}s", refresh=True
        )

@register_cell_magic
def cell_progress(line, cell):
    """Run a cell with a heartbeat bar; inner tqdm bars show countable work."""
    global _ACTIVE_CELL_PROGRESS
    specification = line.strip() or 'Cell progress'
    match = re.fullmatch(r'(\d+)/(\d+)\s+(.+)', specification)
    if match:
        current, total, label = int(match.group(1)), int(match.group(2)), match.group(3)
        initial = current - 1
    else:
        current, total, label, initial = 1, 1, specification, 0
    started = time.monotonic()
    bar = tqdm(
        total=total, initial=initial, desc=label, unit='cell',
        dynamic_ncols=True, leave=True,
    )
    state = {'bar': bar, 'started': started, 'stage': 'starting'}
    _ACTIVE_CELL_PROGRESS = state
    stopped = threading.Event()

    def heartbeat():
        while not stopped.wait(1.0):
            elapsed = time.monotonic() - started
            bar.set_postfix_str(
                f"{state['stage']} | elapsed {elapsed:.0f}s", refresh=True
            )

    worker = threading.Thread(target=heartbeat, daemon=True)
    worker.start()
    succeeded = False
    try:
        cell_stage('running')
        result = get_ipython().run_cell(cell)
        result.raise_error()
        succeeded = True
        return result.result
    finally:
        stopped.set()
        worker.join(timeout=2.0)
        elapsed = time.monotonic() - started
        if succeeded:
            bar.n = current
            bar.set_postfix_str(f"complete | elapsed {elapsed:.0f}s", refresh=True)
        else:
            bar.set_postfix_str(f"failed | elapsed {elapsed:.0f}s", refresh=True)
        bar.close()
        _ACTIVE_CELL_PROGRESS = None

_SETUP_PROGRESS.update(1)
_SETUP_PROGRESS.close()
print('Cell progress helper ready: 14 tracked cells.')

Initialize progress display:   0%|          | 0/1 [00:00<?, ?cell/s]

Cell progress helper ready: 14 tracked cells.


## 1. Mount Drive and point to the project


In [2]:
%%cell_progress 1/14 Mount Drive and locate project
cell_stage('Mounting Google Drive')
from google.colab import drive
drive.mount('/content/drive')

cell_stage('Locating VisualTorque project')
from pathlib import Path
DRIVE_PROJECT = Path('/content/drive/MyDrive/VisualTorque')  # change only this line if needed
candidates = (DRIVE_PROJECT, DRIVE_PROJECT / 'VisualTorque')
PROJECT_ROOT = next(
    (path for path in candidates if (path / 'toolkit' / 'pyproject.toml').is_file()),
    None,
)
assert PROJECT_ROOT is not None, (
    'Could not find toolkit/pyproject.toml. Checked: ' + ', '.join(map(str, candidates))
)
TOOLKIT_ROOT = PROJECT_ROOT / 'toolkit'
print('Project:', PROJECT_ROOT)

Mount Drive and locate project:   0%|          | 0/14 [00:00<?, ?cell/s]

Mounted at /content/drive
Project: /content/drive/MyDrive/VisualTorque


## 2. Install the pinned inference environment

The target models use the current multimodal `AutoProcessor` / `AutoModelForMultimodalLM` interface.


In [3]:
%%cell_progress 2/14 Install and verify environment
import importlib, subprocess, sys

# VisualTorque does not use audio. A stale Colab torchaudio wheel can have a
# different CUDA build from torch and break transformers' AutoProcessor import.
cell_stage('Removing unused TorchAudio')
subprocess.run(
    [sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchaudio'],
    check=False,
)
cell_stage('Installing Transformers dependencies')
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q', '-U',
    'transformers==5.14.0', 'accelerate>=1.10',
    'sentencepiece', 'protobuf', 'huggingface_hub>=0.34', 'tqdm>=4.66'
])
cell_stage('Installing VisualTorque toolkit')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(TOOLKIT_ROOT)])

# Editable installs on Drive are not always added to an already-running kernel.
SRC_ROOT = TOOLKIT_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))
importlib.invalidate_caches()
cell_stage('Checking multimodal imports')
import visual_torque_toolkit

# Fail here with the real environment error, before a large model is loaded.
import torch, torchvision
from transformers import AutoModelForMultimodalLM, AutoProcessor
from visual_torque_toolkit.behavior.adapter import HuggingFaceMultimodalAdapter
required_adapter_features = (
    '_parse_generated_content',
    '_literal_completion_tokens',
)
assert all(hasattr(HuggingFaceMultimodalAdapter, name) for name in required_adapter_features), (
    'Notebook/toolkit version mismatch. Sync the complete toolkit/src folder to Drive, '
    'restart the session, and rerun from the top.'
)
print('torch:', torch.__version__, '| CUDA:', torch.version.cuda)
print('torchvision:', torchvision.__version__)
print('Toolkit imported from:', visual_torque_toolkit.__file__)
print('Multimodal imports: OK')

Install and verify environment:   7%|7         | 1/14 [00:00<?, ?cell/s]

torch: 2.11.0+cu128 | CUDA: 12.8
torchvision: 0.26.0+cu128
Toolkit imported from: /content/drive/MyDrive/VisualTorque/toolkit/src/visual_torque_toolkit/__init__.py
Multimodal imports: OK


## 3. Verify the A100 and authenticate

Create a Colab secret named `HF_TOKEN`.


In [4]:
%%cell_progress 3/14 Verify GPU and Hugging Face access
cell_stage('Checking CUDA device')
import torch
assert torch.cuda.is_available(), 'Select an A100 GPU runtime before continuing.'
print(torch.cuda.get_device_name(0))
print('CUDA memory (GB):', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))

cell_stage('Authenticating with Hugging Face')
from google.colab import userdata
from huggingface_hub import login
try:
    login(token=userdata.get('HF_TOKEN'), add_to_git_credential=False)
    print('Hugging Face login complete.')
except Exception as exc:
    print('No HF_TOKEN used. Public Qwen access may still work. Details:', exc)

Verify GPU and Hugging Face access:  14%|#4        | 2/14 [00:00<?, ?cell/s]

NVIDIA A100-SXM4-40GB
CUDA memory (GB): 39.5
No HF_TOKEN used. Public Qwen access may still work. Details: Secret HF_TOKEN does not exist.


## 4. Build or verify Dataset v1

This creates the full 2,100-scene dataset only when it is absent.


In [5]:
%%cell_progress 4/14 Build, validate, and cache Dataset v1
cell_stage('Checking local and Drive caches')
from visual_torque_toolkit import build_dataset, validate_dataset
from inspect import signature
import shutil
import tempfile
from zipfile import ZIP_DEFLATED, ZipFile

DRIVE_DATA_ROOT = PROJECT_ROOT / 'data'
DATASET_ARCHIVE = DRIVE_DATA_ROOT / 'dataset_v1.zip'
LEGACY_DATASET_DIR = DRIVE_DATA_ROOT / 'dataset_v1'
DEFAULT_RUNTIME_DIR = Path('/content/visualtorque_dataset_v1')
REQUIRED_DATASET_FILES = {
    'dataset_config.json', 'scenes.jsonl', 'counterfactual_pairs.jsonl',
    'questions.jsonl', 'splits.json', 'statistics.json',
}

def call_with_optional_progress(function, *args):
    # New toolkit versions expose show_progress. Falling back keeps an older
    # Drive copy usable; the outer heartbeat still shows elapsed time.
    kwargs = {'show_progress': True} if 'show_progress' in signature(function).parameters else {}
    return function(*args, **kwargs)

if 'show_progress' not in signature(validate_dataset).parameters:
    print('Compatibility mode: toolkit has no show_progress API. Sync toolkit/dataset.py for inner image bars.')

def dataset_looks_complete(root):
    return root.is_dir() and all((root / name).is_file() for name in REQUIRED_DATASET_FILES)

def extract_with_progress(archive, destination):
    with ZipFile(archive, 'r') as bundle:
        members = [member for member in bundle.infolist() if not member.is_dir()]
        destination_resolved = destination.resolve()
        for member in tqdm(members, desc='Extracting cached dataset', unit='file'):
            target = (destination / member.filename).resolve()
            if destination_resolved not in target.parents:
                raise ValueError(f'Unsafe path in dataset archive: {member.filename}')
            bundle.extract(member, destination)

def copytree_with_progress(source, destination):
    file_count = sum(path.is_file() for path in source.rglob('*'))
    with tqdm(total=file_count, desc='Migrating legacy Drive cache', unit='file') as bar:
        def copy_file(source_file, destination_file):
            result = shutil.copy2(source_file, destination_file)
            bar.update(1)
            return result
        shutil.copytree(source, destination, copy_function=copy_file)

def archive_with_progress(source, archive):
    files = sorted(path for path in source.rglob('*') if path.is_file())
    archive.parent.mkdir(parents=True, exist_ok=True)
    incomplete = archive.with_suffix('.zip.incomplete')
    with ZipFile(incomplete, 'w', compression=ZIP_DEFLATED, compresslevel=1) as bundle:
        for path in tqdm(files, desc='Caching dataset to Drive', unit='file'):
            bundle.write(path, path.relative_to(source))
    incomplete.replace(archive)

validation = None
if dataset_looks_complete(DEFAULT_RUNTIME_DIR):
    DATASET_DIR = DEFAULT_RUNTIME_DIR
    cell_stage('Reusing dataset already on local SSD')
else:
    # Fresh sessions use a stable path so legacy run identities can resume.
    # If an interrupted run left that path non-empty, build safely elsewhere.
    if DEFAULT_RUNTIME_DIR.exists():
        scratch_root = Path(tempfile.mkdtemp(prefix='visualtorque_dataset_'))
        DATASET_DIR = scratch_root / 'dataset_v1'
    else:
        DATASET_DIR = DEFAULT_RUNTIME_DIR
    if DATASET_ARCHIVE.is_file():
        cell_stage('Extracting one-file cache from Drive')
        DATASET_DIR.mkdir(parents=True, exist_ok=True)
        extract_with_progress(DATASET_ARCHIVE, DATASET_DIR)
    elif dataset_looks_complete(LEGACY_DATASET_DIR):
        cell_stage('Migrating old expanded Drive cache to local SSD')
        copytree_with_progress(LEGACY_DATASET_DIR, DATASET_DIR)
    else:
        cell_stage('Building 2,100 scenes on local SSD')
        summary = call_with_optional_progress(build_dataset, DATASET_DIR)
        validation = {
            'scene_count': summary.scene_count,
            'pair_count': summary.pair_count,
            'question_count': summary.question_count,
            'image_count': summary.image_count,
        }
        print('Built and internally validated:', validation)

if validation is None:
    cell_stage('Validating local dataset once')
    validation = call_with_optional_progress(validate_dataset, DATASET_DIR)

assert validation['scene_count'] == 2100 and validation['question_count'] == 27000, validation
print('Validated runtime dataset:', validation)

if not DATASET_ARCHIVE.is_file():
    cell_stage('Writing one-file dataset cache to Drive')
    archive_with_progress(DATASET_DIR, DATASET_ARCHIVE)
    print('Created reusable cache:', DATASET_ARCHIVE)

cell_stage('Dataset ready on local SSD')
print('Runtime dataset:', DATASET_DIR)

Build, validate, and cache Dataset v1:  21%|##1       | 3/14 [00:00<?, ?cell/s]

Rendering dataset images:   0%|          | 0/2100 [00:00<?, ?image/s]

Verifying dataset images:   0%|          | 0/2100 [00:00<?, ?image/s]

Built and internally validated: {'scene_count': 2100, 'pair_count': 6427, 'question_count': 27000, 'image_count': 2100}
Validated runtime dataset: {'scene_count': 2100, 'pair_count': 6427, 'question_count': 27000, 'image_count': 2100}


Caching dataset to Drive:   0%|          | 0/2106 [00:00<?, ?file/s]

Created reusable cache: /content/drive/MyDrive/VisualTorque/data/dataset_v1.zip
Runtime dataset: /content/visualtorque_dataset_v1


## 5. Select exactly one target model

Start with Qwen.


In [6]:
%%cell_progress 5/14 Configure target model
cell_stage('Creating model result directory')
# Primary target
MODEL_ID = 'Qwen/Qwen3.5-9B'
BATCH_SIZE = 8

# Second target: uncomment in a fresh runtime after finishing Qwen.
# MODEL_ID = 'google/gemma-4-12B-it'
# BATCH_SIZE = 4

MODEL_SLUG = MODEL_ID.replace('/', '__')
MODEL_RESULT_ROOT = PROJECT_ROOT / 'results' / MODEL_SLUG / 'behavior'
MODEL_RESULT_ROOT.mkdir(parents=True, exist_ok=True)
print(MODEL_ID, MODEL_RESULT_ROOT)

Configure target model:  29%|##8       | 4/14 [00:00<?, ?cell/s]

Qwen/Qwen3.5-9B /content/drive/MyDrive/VisualTorque/results/Qwen__Qwen3.5-9B/behavior


## 6. Load the model in bfloat16

The adapter records the resolved model commit and package versions.


In [7]:
%%cell_progress 6/14 Download and load model
cell_stage('Importing model adapter')
from visual_torque_toolkit.behavior import HuggingFaceMultimodalAdapter

cell_stage('Downloading/loading processor and model weights')
adapter = HuggingFaceMultimodalAdapter(
    MODEL_ID, revision='main', dtype='bfloat16', device_map='auto'
)
cell_stage('Collecting model metadata')
adapter.metadata

Download and load model:  36%|###5      | 5/14 [00:00<?, ?cell/s]

preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/7.76k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/3.13k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/760 [00:00<?, ?it/s]

{'adapter': 'HuggingFaceMultimodalAdapter',
 'model_id': 'Qwen/Qwen3.5-9B',
 'requested_revision': 'main',
 'resolved_revision': 'c202236235762e1c871ad0ccb60c8ee5ba337b9a',
 'dtype': 'bfloat16',
 'device_map': 'auto',
 'parameter_count': 9409813744,
 'torch_version': '2.11.0+cu128',
 'transformers_version': '5.14.0'}

{'adapter': 'HuggingFaceMultimodalAdapter',
 'model_id': 'Qwen/Qwen3.5-9B',
 'requested_revision': 'main',
 'resolved_revision': 'c202236235762e1c871ad0ccb60c8ee5ba337b9a',
 'dtype': 'bfloat16',
 'device_map': 'auto',
 'parameter_count': 9409813744,
 'torch_version': '2.11.0+cu128',
 'transformers_version': '5.14.0'}

## 7. Mandatory smoke test

This scores all six option permutations for eight scenes in both image and symbolic conditions and also generates every answer.


In [8]:
%%cell_progress 7/14 Run mandatory smoke test
cell_stage('Preparing smoke-test examples')
from visual_torque_toolkit.behavior import (
    BehaviorRunConfig, PromptCondition, run_behavior_evaluation
)

smoke_config = BehaviorRunConfig(
    dataset_dir=DATASET_DIR,
    output_dir=MODEL_RESULT_ROOT / '00_smoke',
    model_id=MODEL_ID,
    conditions=(PromptCondition.IMAGE_RULE, PromptCondition.SYMBOLIC_RULE),
    question_kinds=('outcome',),
    batch_size=min(BATCH_SIZE, 4),
    max_scenes=8,
    generation_every=1,
)
cell_stage('Scoring/generating; follow the inner batch bar')
smoke_result = run_behavior_evaluation(smoke_config, adapter)
smoke_result['progress']

Run mandatory smoke test:  43%|####2     | 6/14 [00:00<?, ?cell/s]

VisualTorque inference (96 examples remaining):   0%|          | 0/24 [00:00<?, ?batch/s]

{'expected': 96, 'completed': 96, 'newly_completed': 96}

{'expected': 96, 'completed': 96, 'newly_completed': 96}

In [9]:
%%cell_progress 8/14 Inspect smoke-test reports
cell_stage('Reading smoke-test JSON reports')
import json
def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

smoke_metrics = read_json(MODEL_RESULT_ROOT / '00_smoke' / 'metrics.json')
smoke_options = read_json(MODEL_RESULT_ROOT / '00_smoke' / 'option_robustness.json')
print(json.dumps(smoke_metrics['overall'], indent=2))
print(json.dumps(smoke_options, indent=2)[:4000])

Inspect smoke-test reports:  50%|#####     | 7/14 [00:00<?, ?cell/s]

{
  "count": 96,
  "generation_accuracy_among_parsed": 0.6354166666666666,
  "generation_audit_count": 96,
  "generation_parse_rate": 1.0,
  "likelihood_accuracy": 0.6354166666666666,
  "mean_correct_margin": 0.05208333712653257,
  "mean_latency_seconds": 0.10980297926041693
}
{
  "image_rule": {
    "accuracy_by_gold_label": {
      "A": {
        "count": 16,
        "generation_accuracy_among_parsed": 0.5,
        "generation_audit_count": 16,
        "generation_parse_rate": 1.0,
        "likelihood_accuracy": 0.5,
        "mean_correct_margin": -0.7421875016880222,
        "mean_latency_seconds": 0.10503729112499727
      },
      "B": {
        "count": 16,
        "generation_accuracy_among_parsed": 0.375,
        "generation_audit_count": 16,
        "generation_parse_rate": 1.0,
        "likelihood_accuracy": 0.375,
        "mean_correct_margin": -0.8750000151339918,
        "mean_latency_seconds": 0.1911776155937499
      },
      "C": {
        "count": 16,
        "generati

## 8. Primary full image experiment

This is the headline behavior run: all 2,100 scenes × all six outcome-option permutations = 12,600 scored examples. `generation_every=0` avoids paying generation cost; candidate likelihood is the primary prediction.


In [10]:
%%cell_progress 9/14 Run primary image experiment
cell_stage('Preparing all 12,600 scored examples')
primary_config = BehaviorRunConfig(
    dataset_dir=DATASET_DIR,
    output_dir=MODEL_RESULT_ROOT / '01_primary_image_rule',
    model_id=MODEL_ID,
    conditions=(PromptCondition.IMAGE_RULE,),
    question_kinds=('outcome',),
    batch_size=BATCH_SIZE,
    generation_every=0,
)
cell_stage('Scoring examples; follow the inner batch bar and ETA')
primary_result = run_behavior_evaluation(primary_config, adapter)
primary_result['progress']

Run primary image experiment:  57%|#####7    | 8/14 [00:00<?, ?cell/s]

VisualTorque inference (12600 examples remaining):   0%|          | 0/1575 [00:00<?, ?batch/s]

{'expected': 12600, 'completed': 12600, 'newly_completed': 12600}

{'expected': 12600, 'completed': 12600, 'newly_completed': 12600}

## 9. Greedy-generation format audit

A separate 50-scene audit verifies that free generation follows the one-letter contract.


In [11]:
%%cell_progress 10/14 Run generation-format audit
cell_stage('Preparing generation audit')
audit_config = BehaviorRunConfig(
    dataset_dir=DATASET_DIR,
    output_dir=MODEL_RESULT_ROOT / '02_generation_audit',
    model_id=MODEL_ID,
    conditions=(PromptCondition.IMAGE_RULE,),
    question_kinds=('outcome',),
    batch_size=BATCH_SIZE,
    max_scenes=50,
    generation_every=1,
)
cell_stage('Scoring/generating; follow the inner batch bar')
audit_result = run_behavior_evaluation(audit_config, adapter)
cell_stage('Reading generation-audit metrics')
read_json(MODEL_RESULT_ROOT / '02_generation_audit' / 'metrics.json')['overall']

Run generation-format audit:  64%|######4   | 9/14 [00:00<?, ?cell/s]

VisualTorque inference (300 examples remaining):   0%|          | 0/38 [00:00<?, ?batch/s]

{'count': 300,
 'generation_accuracy_among_parsed': 0.6333333333333333,
 'generation_audit_count': 300,
 'generation_parse_rate': 1.0,
 'likelihood_accuracy': 0.6333333333333333,
 'mean_correct_margin': 0.26166666759798923,
 'mean_latency_seconds': 0.094384589883335}

{'count': 300,
 'generation_accuracy_among_parsed': 0.6333333333333333,
 'generation_audit_count': 300,
 'generation_parse_rate': 1.0,
 'likelihood_accuracy': 0.6333333333333333,
 'mean_correct_margin': 0.26166666759798923,
 'mean_latency_seconds': 0.094384589883335}

## 10. Perception-versus-computation control

This matched subset runs both image and exact symbolic inputs.


In [12]:
%%cell_progress 11/14 Run image-versus-symbolic control
cell_stage('Preparing perception/control examples')
control_config = BehaviorRunConfig(
    dataset_dir=DATASET_DIR,
    output_dir=MODEL_RESULT_ROOT / '03_image_symbolic_control',
    model_id=MODEL_ID,
    conditions=(PromptCondition.IMAGE_RULE, PromptCondition.SYMBOLIC_RULE),
    question_kinds=(
        'outcome', 'left_units', 'right_units',
        'left_distance', 'right_distance'
    ),
    batch_size=BATCH_SIZE,
    scenes_per_stratum=3,
    max_scenes=120,
    generation_every=0,
)
cell_stage('Scoring image and symbolic prompts; follow the inner batch bar')
control_result = run_behavior_evaluation(control_config, adapter)
cell_stage('Reading image-versus-symbolic metrics')
read_json(MODEL_RESULT_ROOT / '03_image_symbolic_control' / 'image_vs_symbolic.json')['overall']

Run image-versus-symbolic control:  71%|#######1  | 10/14 [00:00<?, ?cell/s]

VisualTorque inference (2304 examples remaining):   0%|          | 0/288 [00:00<?, ?batch/s]

{'image_accuracy': 0.6015625,
 'paired_count': 1152,
 'quadrants': {'image_correct__symbolic_correct': 579,
  'image_correct__symbolic_wrong': 114,
  'image_wrong__symbolic_correct': 274,
  'image_wrong__symbolic_wrong': 185},
 'semantic_agreement_rate': 0.6293402777777778,
 'symbolic_accuracy': 0.7404513888888888}

{'image_accuracy': 0.6015625,
 'paired_count': 1152,
 'quadrants': {'image_correct__symbolic_correct': 579,
  'image_correct__symbolic_wrong': 114,
  'image_wrong__symbolic_correct': 274,
  'image_wrong__symbolic_wrong': 185},
 'semantic_agreement_rate': 0.6293402777777778,
 'symbolic_accuracy': 0.7404513888888888}

## 11. Full symbolic outcome control

This repeats the complete 2,100-scene outcome evaluation using exact symbolic scene descriptions.


In [13]:
%%cell_progress 12/14 Run full symbolic outcome control
cell_stage('Preparing 12,600 symbolic outcome examples')
full_symbolic_config = BehaviorRunConfig(
    dataset_dir=DATASET_DIR,
    output_dir=MODEL_RESULT_ROOT / '04_full_symbolic_outcome',
    model_id=MODEL_ID,
    conditions=(PromptCondition.SYMBOLIC_RULE,),
    question_kinds=('outcome',),
    batch_size=BATCH_SIZE,
    generation_every=0,
)
cell_stage('Scoring symbolic scenes; follow the inner batch bar and ETA')
full_symbolic_result = run_behavior_evaluation(full_symbolic_config, adapter)
cell_stage('Reading full symbolic metrics')
read_json(MODEL_RESULT_ROOT / '04_full_symbolic_outcome' / 'metrics.json')['overall']

Run full symbolic outcome control:  79%|#######8  | 11/14 [00:00<?, ?cell/s]

VisualTorque inference (12600 examples remaining):   0%|          | 0/1575 [00:00<?, ?batch/s]

{'count': 12600,
 'generation_accuracy_among_parsed': None,
 'generation_audit_count': 0,
 'generation_parse_rate': None,
 'likelihood_accuracy': 0.6417460317460317,
 'mean_correct_margin': 0.38119543686878704,
 'mean_latency_seconds': 0.02904387604841264}

{'count': 12600,
 'generation_accuracy_among_parsed': None,
 'generation_audit_count': 0,
 'generation_parse_rate': None,
 'likelihood_accuracy': 0.6417460317460317,
 'mean_correct_margin': 0.38119543686878704,
 'mean_latency_seconds': 0.02904387604841264}

## 12. Inspect the reports


In [14]:
%%cell_progress 13/14 Inspect paper-facing reports
cell_stage('Reading five primary reports')
PRIMARY = MODEL_RESULT_ROOT / '01_primary_image_rule'
report_names = [
    'metrics.json', 'metrics_by_problem_type.json', 'option_robustness.json',
    'counterfactual_consistency.json', 'heuristic_agreement.json'
]
for name in tqdm(report_names, desc='Reading reports', unit='report'):
    print('\n###', name)
    print(json.dumps(read_json(PRIMARY / name), indent=2)[:5000])

Inspect paper-facing reports:  86%|########5 | 12/14 [00:00<?, ?cell/s]

Reading reports:   0%|          | 0/5 [00:00<?, ?report/s]


### metrics.json
{
  "by_condition": {
    "image_rule": {
      "count": 12600,
      "generation_accuracy_among_parsed": null,
      "generation_audit_count": 0,
      "generation_parse_rate": null,
      "likelihood_accuracy": 0.5988095238095238,
      "mean_correct_margin": 0.199861111174308,
      "mean_latency_seconds": 0.0942443278878572
    }
  },
  "by_condition_and_question": {
    "image_rule::outcome": {
      "count": 12600,
      "generation_accuracy_among_parsed": null,
      "generation_audit_count": 0,
      "generation_parse_rate": null,
      "likelihood_accuracy": 0.5988095238095238,
      "mean_correct_margin": 0.199861111174308,
      "mean_latency_seconds": 0.0942443278878572
    }
  },
  "by_question_kind": {
    "outcome": {
      "count": 12600,
      "generation_accuracy_among_parsed": null,
      "generation_audit_count": 0,
      "generation_parse_rate": null,
      "likelihood_accuracy": 0.5988095238095238,
      "mean_correct_margin": 0.199861111174308,


## 13. Release GPU memory before switching models

For the cleanest comparison, start a fresh Colab runtime for the second model.


In [15]:
%%cell_progress 14/14 Release GPU memory
cell_stage('Deleting model and collecting Python objects')
import gc
del adapter
gc.collect()
cell_stage('Emptying CUDA cache')
torch.cuda.empty_cache()
print('GPU cache released. A fresh runtime is still recommended for the next model.')

Release GPU memory:  93%|#########2| 13/14 [00:00<?, ?cell/s]

GPU cache released. A fresh runtime is still recommended for the next model.
